In [8]:
import torch
import numpy as np
import scipy.sparse as sp
from collections import defaultdict
from tqdm import tqdm
import os

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [10]:
def load_data(train_file, test_file):

    train_dict = defaultdict(set)
    test_dict = defaultdict(set)

    with open(train_file) as f:
        for line in f:
            parts = line.strip().split()
            u = int(parts[0])
            items = list(map(int, parts[1:]))
            train_dict[u].update(items)

    with open(test_file) as f:
        for line in f:
            parts = line.strip().split()
            u = int(parts[0])
            items = list(map(int, parts[1:]))
            test_dict[u].update(items)

    n_users = max(train_dict.keys()) + 1
    n_items = max(max(items) for items in train_dict.values()) + 1

    return train_dict, test_dict, n_users, n_items

In [11]:
def build_norm_adj(train_dict, n_users, n_items):

    rows, cols = [], []

    for u, items in train_dict.items():
        for i in items:
            rows.append(u)
            cols.append(i + n_users)

    n = n_users + n_items

    data = np.ones(len(rows))
    adj = sp.coo_matrix((data, (rows, cols)), shape=(n, n))

    adj = adj + adj.T

    rowsum = np.array(adj.sum(axis=1)).flatten()
    d_inv = np.power(rowsum, -0.5)
    d_inv[np.isinf(d_inv)] = 0

    d_mat = sp.diags(d_inv)
    norm_adj = d_mat @ adj @ d_mat

    return norm_adj

In [12]:
import torch.nn as nn

class LightGCN(nn.Module):

    def __init__(self, n_users, n_items, embed_dim, n_layers, A_norm):
        super().__init__()

        self.n_users = n_users
        self.n_items = n_items
        self.n_layers = n_layers
        self.A_norm = A_norm

        self.user_emb = nn.Embedding(n_users, embed_dim)
        self.item_emb = nn.Embedding(n_items, embed_dim)

        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

    def propagate(self):

        all_emb = torch.cat([self.user_emb.weight,
                             self.item_emb.weight])

        embs = [all_emb]

        for _ in range(self.n_layers):
            all_emb = torch.sparse.mm(self.A_norm, all_emb)
            embs.append(all_emb)

        embs = torch.stack(embs, dim=1)
        final_emb = torch.mean(embs, dim=1)

        users, items = torch.split(final_emb,
                                   [self.n_users, self.n_items])

        return users, items

    def get_embeddings(self):
        return self.propagate()

In [13]:
@torch.no_grad()
def evaluate(model, train_dict, test_dict, n_users, n_items, k=20):

    model.eval()

    u_emb, i_emb = model.get_embeddings()
    scores = torch.matmul(u_emb, i_emb.t())

    recall = 0
    ndcg = 0
    n_eval = 0

    for u in tqdm(test_dict):

        if len(test_dict[u]) == 0:
            continue

        user_scores = scores[u].cpu().numpy()

        train_items = train_dict[u]
        user_scores[list(train_items)] = -np.inf

        topk = np.argsort(-user_scores)[:k]

        gt_items = test_dict[u]

        hits = [1 if i in gt_items else 0 for i in topk]

        recall += sum(hits) / len(gt_items)

        dcg = 0
        for idx, h in enumerate(hits):
            if h:
                dcg += 1 / np.log2(idx + 2)

        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(gt_items), k)))

        ndcg += dcg / idcg if idcg > 0 else 0

        n_eval += 1

    return recall / n_eval, ndcg / n_eval

In [15]:
def convert_sp_mat_to_sp_tensor(matrix):

    matrix = matrix.tocoo().astype(np.float32)

    indices = torch.from_numpy(
        np.vstack((matrix.row, matrix.col)).astype(np.int64)
    )

    values = torch.from_numpy(matrix.data)

    shape = torch.Size(matrix.shape)

    return torch.sparse.FloatTensor(indices, values, shape)

In [21]:
TRAIN_FILE = r"F:\SEM-6\ELECTIVES\DEPARTMENT-ELECTIVES\Recommendation Systems\Project\LightGCN-master\Data\gowalla\train.txt"
TEST_FILE  = r"F:\SEM-6\ELECTIVES\DEPARTMENT-ELECTIVES\Recommendation Systems\Project\LightGCN-master\Data\gowalla\test.txt"

EMBED_DIM = 64
N_LAYERS = 3
TOPK = 20

train_dict, test_dict, n_users, n_items = load_data(TRAIN_FILE, TEST_FILE)

A_norm = build_norm_adj(train_dict, n_users, n_items)
A_norm = convert_sp_mat_to_sp_tensor(A_norm).to(device)

model = LightGCN(n_users, n_items, EMBED_DIM, N_LAYERS, A_norm).to(device)

ckpt = torch.load("k3_results/lightgcn_outputs/best_model.pt", map_location=device)
model.load_state_dict(ckpt["model_state_dict"])

print("Model loaded from epoch:", ckpt["epoch"])
print("Recall:", ckpt["recall"])
print("NDCG:", ckpt["ndcg"])

Model loaded from epoch: 100
Recall: 0.170044444850745
NDCG: 0.14201397147464065


In [23]:
model = LightGCN(n_users, n_items, EMBED_DIM, N_LAYERS, A_norm).to(device)

ckpt = torch.load("k3_results/lightgcn_outputs/best_model.pt", map_location=device)

model.load_state_dict(ckpt["model_state_dict"])

print("Loaded model from epoch:", ckpt["epoch"])

recall, ndcg = evaluate(
    model,
    train_dict,
    test_dict,
    n_users,
    n_items,
    k=20
)

print("Test Recall@20:", recall)
print("Test NDCG@20:", ndcg)

Loaded model from epoch: 100


100%|██████████| 29858/29858 [01:25<00:00, 347.92it/s]

Test Recall@20: 0.16630807072787937
Test NDCG@20: 0.14201397147464065
